In [3]:
"""
generate_negatives.py – Generate non-compliant (negative) training examples
by injecting known Client X style-guide violations into positive pairs.
 
Usage:
    python scripts/generate_negatives.py
 
Input:
    processed/sentence_pairs.jsonl   (positive pairs)
 
Output:
    processed/negatives.jsonl        (label = 0)
 
Violation tiers
---------------
Tier 1 – Unconditional string substitutions (safe to automate):
    V01  percent → per cent
    V02  website → web site
    V03  SEK X million → X Mkr
    V04  m2 / m² → square meters
    V05  CPI → KPI
    V06  yield → direct yield  (property; excludes dividend yield)
    V07  occupancy rate → occupancy ratio
    V07b occupancy rate → letting ratio
    V08  divestment → disposal
    V14  weighted average lease expiry → average contract period
    V15  land properties → land holdings
    V16  building frames → carcasses
    V17  new share issue → rights issue
    V18  property tax value → taxable values
    V19  Profit from investments in → Profit from participations in
    V20  Equity ratio → Equity/assets ratio
    V21  EPRA performance measures → EPRA key figures
    V22  Net profit for the year → Profit for the year
    V23  Foreign currency transactions and balances →
         Foreign currency transactions and balance sheet items
    V24  Board fees → Directors' fees
    V25  valuation hierarchy → measurement hierarchy
    V26  capitalised interest → capitalised interest rate
    V27  cold storage → refrigeration and freezing facility
    V28  live-streamed → presented online
    V29  Acquisitions, investments and divestments →
         Acquisitions, investments and disposals
    V30  key performance indicators → key financial figures
 
Tier 2 – Context-guarded (only inject when a guard phrase is present):
    V09  material accounting policy information → significant accounting policies
    V10  Parent Company financial statements → Parent Company's financial statements
    V11  Nomination Committee → nomination committee  (capitalisation)
    V12  Remuneration Committee → remuneration committee
    V13  Audit Committee → audit committee
"""
 
import json
import re
import random
from pathlib import Path
from dataclasses import dataclass
 
# ── Config ────────────────────────────────────────────────────────────────────
 
RANDOM_SEED = 42
IN_FILE  = Path("processed/sentence_pairs.jsonl")
OUT_FILE = Path("processed/negatives.jsonl")
 
# Max negatives per rule — keeps dataset balanced
MAX_PER_RULE = 2000
 
 
# ── Violation definitions ─────────────────────────────────────────────────────
 
@dataclass
class Violation:
    vid: str
    desc: str
    tier: int
    guard: str | None
    pattern: re.Pattern
    replacement: str
 
 
def build_violations() -> list[Violation]:
    return [
 
        # ══════════════════════════════════════════════════════════════════
        # TIER 1 — unconditional
        # ══════════════════════════════════════════════════════════════════
 
        Violation(
            vid="V01",
            desc="'percent' → forbidden 'per cent'",
            tier=1, guard=None,
            pattern=re.compile(r"\bpercent\b", re.IGNORECASE),
            replacement="per cent",
        ),
        Violation(
            vid="V02",
            desc="'website' → forbidden 'web site'",
            tier=1, guard=None,
            pattern=re.compile(r"\bwebsite\b", re.IGNORECASE),
            replacement="web site",
        ),
        Violation(
            vid="V03",
            desc="Currency: 'SEK X million' → forbidden 'X Mkr'",
            tier=1, guard=None,
            pattern=re.compile(r"SEK\s+(-?\d[\d,\.]*)\s+million", re.IGNORECASE),
            replacement=r"\1 Mkr",
        ),
        Violation(
            vid="V04",
            desc="Area unit: 'm²/m2' → forbidden 'square meters'",
            tier=1, guard=None,
            pattern=re.compile(r"\bm²\b|\bm2\b"),
            replacement="square meters",
        ),
        Violation(
            vid="V05",
            desc="'CPI' (consumer price index) → forbidden 'KPI'",
            tier=1, guard=None,
            pattern=re.compile(r"\bCPI\b"),
            replacement="KPI",
        ),
        Violation(
            vid="V06",
            desc="'yield' → forbidden 'direct yield' (excludes dividend yield)",
            tier=1, guard=None,
            # Negative lookbehind: don't match if preceded by 'direct' or 'dividend'
            pattern=re.compile(r"(?<!direct\s)(?<!dividend\s)\byield\b", re.IGNORECASE),
            replacement="direct yield",
        ),
        Violation(
            vid="V07",
            desc="'occupancy rate' → forbidden 'occupancy ratio'",
            tier=1, guard=None,
            pattern=re.compile(r"\boccupancy rate\b", re.IGNORECASE),
            replacement="occupancy ratio",
        ),
        Violation(
            vid="V07b",
            desc="'occupancy rate' → forbidden 'letting ratio'",
            tier=1, guard=None,
            pattern=re.compile(r"\boccupancy rate\b", re.IGNORECASE),
            replacement="letting ratio",
        ),
        Violation(
            vid="V08",
            desc="'divestment' → forbidden 'disposal'",
            tier=1, guard=None,
            pattern=re.compile(r"\bdivestment\b", re.IGNORECASE),
            replacement="disposal",
        ),
        Violation(
            vid="V14",
            desc="'weighted average lease expiry' → forbidden 'average contract period'",
            tier=1, guard=None,
            pattern=re.compile(r"\bweighted average lease expiry\b", re.IGNORECASE),
            replacement="average contract period",
        ),
        Violation(
            vid="V15",
            desc="'land properties' → forbidden 'land holdings'",
            tier=1, guard=None,
            pattern=re.compile(r"\bland properties\b", re.IGNORECASE),
            replacement="land holdings",
        ),
        Violation(
            vid="V16",
            desc="'building frame(s)' → forbidden 'carcass(es)'",
            tier=1, guard=None,
            pattern=re.compile(r"\bbuilding frames?\b", re.IGNORECASE),
            replacement="carcasses",
        ),
        Violation(
            vid="V17",
            desc="'new share issue' → forbidden 'rights issue'",
            tier=1, guard=None,
            pattern=re.compile(r"\bnew share issue\b", re.IGNORECASE),
            replacement="rights issue",
        ),
        Violation(
            vid="V18",
            desc="'property tax value' → forbidden 'taxable values'",
            tier=1, guard=None,
            pattern=re.compile(r"\bproperty tax value\b", re.IGNORECASE),
            replacement="taxable values",
        ),
        Violation(
            vid="V19",
            desc="'Profit from investments in' → forbidden 'Profit from participations in'",
            tier=1, guard=None,
            pattern=re.compile(r"\bProfit from investments in\b", re.IGNORECASE),
            replacement="Profit from participations in",
        ),
        Violation(
            vid="V20",
            desc="'Equity ratio' → forbidden 'Equity/assets ratio'",
            tier=1, guard=None,
            pattern=re.compile(r"\bEquity ratio\b", re.IGNORECASE),
            replacement="Equity/assets ratio",
        ),
        Violation(
            vid="V21",
            desc="'EPRA performance measures' → forbidden 'EPRA key figures'",
            tier=1, guard=None,
            pattern=re.compile(r"\bEPRA performance measures\b", re.IGNORECASE),
            replacement="EPRA key figures",
        ),
        Violation(
            vid="V22",
            desc="'Net profit for the year' → forbidden 'Profit for the year'",
            tier=1, guard=None,
            pattern=re.compile(r"\bNet profit for the year\b", re.IGNORECASE),
            replacement="Profit for the year",
        ),
        Violation(
            vid="V23",
            desc="'Foreign currency transactions and balances' → forbidden '... balance sheet items'",
            tier=1, guard=None,
            pattern=re.compile(
                r"\bForeign currency transactions and balances\b", re.IGNORECASE
            ),
            replacement="Foreign currency transactions and balance sheet items",
        ),
        Violation(
            vid="V24",
            desc="'Board fees' → forbidden 'Directors' fees'",
            tier=1, guard=None,
            pattern=re.compile(r"\bBoard fees\b", re.IGNORECASE),
            replacement="Directors' fees",
        ),
        Violation(
            vid="V25",
            desc="'valuation hierarchy' → forbidden 'measurement hierarchy'",
            tier=1, guard=None,
            pattern=re.compile(r"\bvaluation hierarchy\b", re.IGNORECASE),
            replacement="measurement hierarchy",
        ),
        Violation(
            vid="V26",
            desc="'capitalised interest' → forbidden 'capitalised interest rate'",
            tier=1, guard=None,
            # Negative lookahead: don't match if 'rate' already follows
            pattern=re.compile(r"\bcapitalised interest\b(?!\s+rate)", re.IGNORECASE),
            replacement="capitalised interest rate",
        ),
        Violation(
            vid="V27",
            desc="'cold storage' → forbidden 'refrigeration and freezing facility'",
            tier=1, guard=None,
            pattern=re.compile(r"\bcold storage\b", re.IGNORECASE),
            replacement="refrigeration and freezing facility",
        ),
        Violation(
            vid="V28",
            desc="'live-streamed' → forbidden 'presented online'",
            tier=1, guard=None,
            pattern=re.compile(r"\blive-streamed\b", re.IGNORECASE),
            replacement="presented online",
        ),
        Violation(
            vid="V29",
            desc="'Acquisitions, investments and divestments' → forbidden '... disposals'",
            tier=1, guard=None,
            pattern=re.compile(
                r"\bAcquisitions, investments and divestments\b", re.IGNORECASE
            ),
            replacement="Acquisitions, investments and disposals",
        ),
        Violation(
            vid="V30",
            desc="'key performance indicators' → forbidden 'key financial figures'",
            tier=1, guard=None,
            pattern=re.compile(r"\bkey performance indicators\b", re.IGNORECASE),
            replacement="key financial figures",
        ),
 
        # ══════════════════════════════════════════════════════════════════
        # TIER 2 — context-guarded
        # ══════════════════════════════════════════════════════════════════
 
        Violation(
            vid="V09",
            desc="IAS 1: 'material accounting policy information' → 'significant accounting policies'",
            tier=2,
            guard=r"accounting polic",
            pattern=re.compile(
                r"material\s+accounting\s+policy\s+information", re.IGNORECASE
            ),
            replacement="significant accounting policies",
        ),
        Violation(
            vid="V10",
            desc="'Parent Company financial statements' → forbidden possessive form",
            tier=2,
            guard=r"Parent Company",
            pattern=re.compile(r"Parent Company financial statements", re.IGNORECASE),
            replacement="Parent Company's financial statements",
        ),
        Violation(
            vid="V11",
            desc="'Nomination Committee' → wrong lowercase",
            tier=2,
            guard=r"Nomination Committee",
            pattern=re.compile(r"\bNomination Committee\b"),
            replacement="nomination committee",
        ),
        Violation(
            vid="V12",
            desc="'Remuneration Committee' → wrong lowercase",
            tier=2,
            guard=r"Remuneration Committee",
            pattern=re.compile(r"\bRemuneration Committee\b"),
            replacement="remuneration committee",
        ),
        Violation(
            vid="V13",
            desc="'Audit Committee' → wrong lowercase",
            tier=2,
            guard=r"Audit Committee",
            pattern=re.compile(r"\bAudit Committee\b"),
            replacement="audit committee",
        ),
    ]
 
 
# ── Inline tag stripper ───────────────────────────────────────────────────────
 
TAG_RE = re.compile(r"[\[\{]\d+[\]\}]")
 
def strip_tags(text: str) -> str:
    return TAG_RE.sub("", text).strip()
 
 
# ── Core injection logic ──────────────────────────────────────────────────────
 
def try_inject(translation: str, v: Violation) -> str | None:
    if v.tier == 2 and v.guard:
        if not re.search(v.guard, translation):
            return None
    if not v.pattern.search(translation):
        return None
    mutated = v.pattern.sub(v.replacement, translation, count=1)
    if mutated == translation:
        return None
    return mutated
 
 
# ── Main ──────────────────────────────────────────────────────────────────────
 
def main():
    random.seed(RANDOM_SEED)
    violations = build_violations()
 
    positives = []
    with IN_FILE.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                positives.append(json.loads(line))
 
    print(f"Loaded {len(positives)} positive pairs from {IN_FILE}")
    random.shuffle(positives)
 
    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    totals: dict[str, int] = {v.vid: 0 for v in violations}
    written = 0
 
    with OUT_FILE.open("w", encoding="utf-8") as out:
        for v in violations:
            count = 0
            for rec in positives:
                if count >= MAX_PER_RULE:
                    break
                translation_raw = rec.get("translation", "")
                translation = strip_tags(translation_raw)
                if not translation:
                    continue
                mutated = try_inject(translation, v)
                if mutated is None:
                    continue
 
                negative = {
                    **rec,
                    "label": 0,
                    "violation_id": v.vid,
                    "violation_tier": v.tier,
                    "violation_desc": v.desc,
                    "original_en": translation,
                    "translation": mutated,
                }
                out.write(json.dumps(negative, ensure_ascii=False) + "\n")
                count += 1
                written += 1
 
            totals[v.vid] = count
            tier_label = f"T{v.tier}"
            print(f"  {v.vid:<5} ({tier_label}) – {count:>5} negatives  |  {v.desc}")
 
    print(f"\nTotal negatives written: {written} → {OUT_FILE}")
    print("\nViolation breakdown:")
    for vid, n in totals.items():
        bar = "█" * (n // 50)
        print(f"  {vid}: {n:>5}  {bar}")
 
 
if __name__ == "__main__":
    main()

Loaded 17103 positive pairs from processed/sentence_pairs.jsonl
  V01   (T1) –  1017 negatives  |  'percent' → forbidden 'per cent'
  V02   (T1) –    73 negatives  |  'website' → forbidden 'web site'
  V03   (T1) –   950 negatives  |  Currency: 'SEK X million' → forbidden 'X Mkr'
  V04   (T1) –   419 negatives  |  Area unit: 'm²/m2' → forbidden 'square meters'
  V05   (T1) –    27 negatives  |  'CPI' (consumer price index) → forbidden 'KPI'
  V06   (T1) –   156 negatives  |  'yield' → forbidden 'direct yield' (excludes dividend yield)
  V07   (T1) –    32 negatives  |  'occupancy rate' → forbidden 'occupancy ratio'
  V07b  (T1) –    32 negatives  |  'occupancy rate' → forbidden 'letting ratio'
  V08   (T1) –    17 negatives  |  'divestment' → forbidden 'disposal'
  V14   (T1) –    27 negatives  |  'weighted average lease expiry' → forbidden 'average contract period'
  V15   (T1) –    17 negatives  |  'land properties' → forbidden 'land holdings'
  V16   (T1) –     4 negatives  |  'buil